# 03 — Automatic Post-training Statistics, XAI, Operational Analysis, and Visualization

This notebook is designed to run **after the training notebooks have finished**. It automatically discovers the benchmark Excel workbooks and their companion compressed prediction/model artifacts.

Excel is used for human-readable summaries, HPO selections, and reporting tables. Dependence-aware inference and XAI require origin-by-horizon prediction arrays and trained models; those objects are therefore loaded automatically from the same result directory rather than forcing millions of values into spreadsheet cells.

No model selection is performed in this notebook.

## 1. Setup and automatic workbook discovery

In [ ]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIG_DIR = OUTPUT_DIR / "posthoc_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTHONHASHSEED", "42")

from lash_revision_core import ExperimentConfig, dataset_output_dir, load_prediction
from lash_revision_analysis import (
    load_excel_results, horizon_paired_tests,
    run_operational_analysis, temporal_robustness_frame,
    lash_internal_diagnostics, integrated_gradients_lash,
    grouped_permutation_lash, tree_shap_lightgbm,
)
from lash_hardware_optimized import apply_hardware_patch
from lash_per_dataset_hpo import PerDatasetHPOPolicy, run_priority_statistical_tests

apply_hardware_patch()
policy = PerDatasetHPOPolicy(computational_budget_hours=48.0)

config = ExperimentConfig(
    data_root=DATA_DIR,
    output_root=OUTPUT_DIR,
    dataset_keys=("CLUSTER_1", "CLUSTER_2", "BDG_EDU", "BDG_DORM"),
    run_profile="paper",
    weather_mode="historical_only",
    hpo_seeds=policy.hpo_seeds,
    final_refit_seeds=policy.lash_final_seeds,
    primary_hpo_repeats=2,
    external_hpo_repeats=2,
    hpo_trials_per_dimension=1,
    hpo_min_trials=3,
    hpo_max_trials=5,
    max_epochs=policy.max_epochs,
    early_stopping_patience=policy.patience,
    save_models=True,
)

excel_results = load_excel_results(OUTPUT_DIR)
print("Discovered benchmark workbooks:", list(excel_results))
for key, sheets in excel_results.items():
    print("=" * 18, key, "=" * 18)
    display(sheets["Benchmark_Summary"].head(20))


## 2. Dependence-aware paired inference

Hourly rolling 24-step forecast origins overlap, so origin-level losses are serially dependent. The first-revision inferential analysis uses the prespecified focal comparator family and:

- Newey–West HAC lags `24, 72, 168, 336`;
- circular block-bootstrap lengths `24, 72, 168, 336`;
- 1,000 bootstrap replicates for the first-revision analysis;
- seed-matched paired losses from the three final stochastic refits;
- a hierarchical bootstrap that resamples training seeds and temporal blocks;
- Holm multiplicity adjustment;
- paired-loss ACF values through lag 336.

Notebook 04 contains the separate second-round ten-seed / 5,000-replicate confirmatory extension.


In [ ]:
# Primary reviewer-facing inferential family:
# three matched stochastic refits + dependence-aware 24/72/168/336-h tests.
stat_results = run_priority_statistical_tests(config, policy)
for key, table in stat_results.items():
    print(key)
    sort_cols = [c for c in ("method", "setting", "p_holm") if c in table.columns]
    display(table.sort_values(sort_cols).head(60) if sort_cols else table.head(60))


### Optional horizon-specific inference

Horizon-level testing is intentionally separate from the overall 24-hour trajectory test. The example below evaluates a prespecified set of strong comparators and adjusts the 24 horizon p-values within each seed.

In [ ]:
HORIZON_COMPARATORS = ["RIDGE", "LIGHTGBM", "XGBOOST", "CATBOOST", "GRU", "TCN", "TRANSFORMER"]
horizon_tables = []
for key in config.dataset_keys:
    for comparator in HORIZON_COMPARATORS:
        try:
            t = horizon_paired_tests(config, key, comparator, hac_lag=168)
            t.insert(0, "dataset", key)
            horizon_tables.append(t)
        except FileNotFoundError:
            pass

horizon_tests = pd.concat(horizon_tables, ignore_index=True) if horizon_tables else pd.DataFrame()
display(horizon_tests.head())

## 3. Temporal robustness and BEMS-oriented peak sensitivity

These analyses do not alter the forecasts. They ask whether lower forecasting error also improves decision-relevant signals:

- rolling 24-hour trajectory peak-magnitude error;
- trajectory peak-timing error;
- top-three peak-hour hit rate;
- recall of 90th- and 95th-percentile high-load periods;
- a representative demand-response sensitivity using the predicted top `1, 2, 4` hours and curtailment fractions of `5%` and `10%`.

The curtailment calculation is a forecasting-to-decision sensitivity analysis, not a complete battery/HVAC optimizer and should be described accordingly.

In [ ]:
operational = run_operational_analysis(config)
for key, table in operational.items():
    print(key)
    display(table.sort_values(["model", "seed"]).head(40))

## 4. HPO convergence diagnostics

For each independent HPO repeat, the best-so-far validation score is reconstructed from the saved trial table. These curves provide a direct visual check of whether the search budget was still producing major improvements near its endpoint.

In [ ]:
def convergence_frame(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    value_col = "value"
    df = df[np.isfinite(pd.to_numeric(df[value_col], errors="coerce"))].copy()
    df[value_col] = pd.to_numeric(df[value_col])
    df["best_so_far"] = df.groupby("hpo_seed")[value_col].cummin()
    return df

for dataset_key in config.dataset_keys:
    table_dir = OUTPUT_DIR / "benchmark" / dataset_key / "tables"
    for path in sorted(table_dir.glob("*_hpo_trials.csv")):
        df = convergence_frame(path)
        if df.empty:
            continue
        fig, ax = plt.subplots(figsize=(6.2, 4.0))
        for seed, g in df.groupby("hpo_seed"):
            ax.plot(np.arange(1, len(g)+1), g["best_so_far"], label=f"HPO seed {seed}")
        ax.set_xlabel("Completed trial")
        ax.set_ylabel("Best validation selection score (%)")
        ax.set_title(f"{dataset_key} — {path.stem.replace('_hpo_trials','')}")
        ax.legend(frameon=False, fontsize=8)
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"{dataset_key}_{path.stem}_convergence.png", dpi=600, bbox_inches="tight")
        plt.close(fig)

## 5. LASH internal diagnostics and Integrated Gradients

The model-level interpretation is intentionally multi-view:

- historical and future feature-gate summaries;
- learned temporal-pooling profiles across the 168-hour lookback;
- Integrated Gradients at horizons `t+1`, `t+12`, and `t+24`;
- grouped permutation of the **complete** routed LASH pipeline.

Attributions are interpreted as model diagnostics rather than causal effects.

In [ ]:
# Same diagnostic contract on all four datasets.
# For BDG datasets, nonlinear diagnostics characterize the frozen candidate
# sequential expert when the final router selects Ridge-only.
XAI_DATASETS = ("CLUSTER_1", "CLUSTER_2", "BDG_EDU", "BDG_DORM")
xai_tables = {}
for key in XAI_DATASETS:
    internal = lash_internal_diagnostics(config, key, seed=42, sample_origins=256)
    ig = integrated_gradients_lash(
        config, key, seed=42, horizons=(1,12,24),
        sample_origins=128, n_steps=32
    )
    perm = grouped_permutation_lash(
        config, key, seed=42, sample_origins=256, repetitions=5
    )
    xai_tables[key] = {
        **internal,
        "integrated_gradients": ig,
        "grouped_permutation": perm,
    }
    print(key)
    display(internal["past_gates"].head(15))
    display(
        ig.sort_values(
            ["horizon", "mean_abs_IG"], ascending=[True, False]
        ).groupby("horizon").head(10)
    )
    display(perm)


## 6. Tree-model SHAP

LightGBM is explained separately with TreeSHAP at horizons 1, 12, and 24. This provides continuity with the earlier tree-ensemble studies while keeping the current forecasting protocol unchanged.

In [ ]:
shap_tables = {}
for key in XAI_DATASETS:
    shap_table = tree_shap_lightgbm(config, key, seed=42, horizons=(1,12,24), sample_origins=256)
    shap_tables[key] = shap_table
    print(key)
    display(shap_table.groupby("horizon").head(15))

## 7. Publication-ready summary figures

The plots below use a consistent font, axis convention, precision, and 600-dpi export. The same style can be reused for all final manuscript figures.

In [ ]:
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
})

for key, sheets in excel_results.items():
    summary = sheets["Benchmark_Summary"].sort_values("selection_score_mean")
    fig, ax = plt.subplots(figsize=(7.2, 5.2))
    ax.barh(summary["model"], summary["selection_score_mean"], xerr=summary["selection_score_std"].fillna(0))
    ax.invert_yaxis()
    ax.set_xlabel("Composite score (%)")
    ax.set_title(f"{key}: direct 24-step benchmark")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{key}_benchmark_composite.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{key}_benchmark_composite.pdf", bbox_inches="tight")
    plt.close(fig)

    # Rolling error for LASH seed 42.
    try:
        payload = load_prediction(dataset_output_dir(config, key), "LASH", 42)
        tr = temporal_robustness_frame(payload)
        fig, ax = plt.subplots(figsize=(7.2, 3.8))
        ax.plot(tr["forecast_origin"], tr["rolling_30d_NMAE"])
        ax.set_ylabel("Rolling 30-day origin NMAE (%)")
        ax.set_xlabel("Forecast origin")
        ax.set_title(f"{key}: temporal test robustness")
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"{key}_rolling_30d_error.png", dpi=600, bbox_inches="tight")
        fig.savefig(FIG_DIR / f"{key}_rolling_30d_error.pdf", bbox_inches="tight")
        plt.close(fig)
    except FileNotFoundError:
        pass

## 8. Consolidated post-training Excel workbook

This workbook collects compact tables that can be inspected without retraining. Full origin-level predictions remain in compressed artifacts to preserve numeric fidelity and avoid spreadsheet size limitations.

In [ ]:
posthoc_path = OUTPUT_DIR / "03_posthoc_analysis_summary.xlsx"
with pd.ExcelWriter(posthoc_path, engine="openpyxl") as writer:
    for key, sheets in excel_results.items():
        sheets["Benchmark_Summary"].to_excel(writer, sheet_name=f"{key[:20]}_Bench", index=False)
    for key, table in stat_results.items():
        table.to_excel(writer, sheet_name=f"{key[:20]}_Stats", index=False)
    if not horizon_tests.empty:
        horizon_tests.to_excel(writer, sheet_name="Horizon_HAC", index=False)
    for key, table in operational.items():
        table.to_excel(writer, sheet_name=f"{key[:20]}_Peak", index=False)
    for key, table in shap_tables.items():
        table.to_excel(writer, sheet_name=f"{key[:20]}_SHAP", index=False)
    for key, tables in xai_tables.items():
        tables["grouped_permutation"].to_excel(writer, sheet_name=f"{key[:16]}_Perm", index=False)
        tables["integrated_gradients"].to_excel(writer, sheet_name=f"{key[:16]}_IG", index=False)

print("Saved:", posthoc_path)
print("Figures:", FIG_DIR)